# 第二十七课:DPO——偏好直接优化

讲义:[27-dpo.md](27-dpo.md)。RLHF 目标的闭式解 → DPO 损失;mask 泛化为逐 token 权重,反向机器零改动;含符号翻转战争故事与「偏好分离器 ≠ 知识注入器」主实验。

In [ ]:
import math

import time

import numpy as np



# ===== 第 20 课 MiniGPT 原码(含 generate) =====

def softmax(x, axis=-1):

    m = np.max(x, axis=axis, keepdims=True)

    e = np.exp(x - m)

    return e / e.sum(axis=axis, keepdims=True)



class MiniGPT:

    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):

        rng = np.random.default_rng(seed)

        s = 0.1

        self.E = rng.normal(size=(vocab, d)) * 0.5

        self.dh = d // n_heads

        self.H, self.D, self.NL = n_heads, d, n_layers

        self.layers = [{

            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,

            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,

            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,

            'W3': rng.normal(size=(d, 4*d))*s*0.8,

        } for _ in range(n_layers)]

        self.head = rng.normal(size=(d, vocab)) * s



    def rms(self, x, eps=1e-5):

        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)



    def rope(self, x, pos):

        half = x.shape[-1] // 2

        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))

        c, s = np.cos(ang), np.sin(ang)

        x1, x2 = x[..., :half], x[..., half:]

        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)



    def _attn(self, x, pos, W, cache):

        n = x.shape[0]

        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']

        if cache is not None:

            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])

        T = k.shape[0]

        mask = np.triu(np.ones((n, T), bool), T - n + 1)

        outs = []

        for h in range(self.H):

            sl = slice(h*self.dh, (h+1)*self.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)

            s = np.where(mask, -np.inf, s)

            outs.append(softmax(s) @ v[:, sl])

        return np.concatenate(outs, -1) @ W['Wo'], (k, v)



    def _ffn(self, x, W):

        h = x @ W['W1']

        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']



    def forward(self, ids, cache=None):

        pos0 = cache[0][0].shape[0] if cache else 0

        x = self.E[list(ids)]

        new_cache = []

        for i, W in enumerate(self.layers):

            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,

                               cache[i] if cache else None)

            x = x + a

            x = x + self._ffn(self.rms(x), W)

            new_cache.append(kv)

        logits = self.rms(x) @ self.head

        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'

        return logits, new_cache



    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):

        rng = rng or np.random.default_rng()

        logits, cache = self.forward(ids)

        out = []

        for _ in range(n):

            p = softmax(logits[-1] / T)

            if top_p < 1.0:

                order = np.argsort(-p); cum = np.cumsum(p[order])

                keep = order[:np.searchsorted(cum, top_p)+1]

                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()

            nxt = int(rng.choice(len(p), p=p/p.sum()))

            out.append(nxt)

            logits, cache = self.forward([nxt], cache)

        return list(ids) + out

In [ ]:
# ===== 第 23 课反向 + 本课 mask 扩展(合并为一个函数;mask=None 即 23 课原版) =====

def rms_backward(dy, x, eps=1e-5):

    r = np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    y = x / r

    return (dy - y * (dy * y).mean(-1, keepdims=True)) / r



def rope_backward(dy, pos, dim):

    half = dim // 2

    ang = pos[:, None] / (10000 ** (np.arange(half)*2/dim))

    c, s = np.cos(ang), np.sin(ang)

    dy1, dy2 = dy[..., :half], dy[..., half:]

    return np.concatenate([dy1*c + dy2*s, -dy1*s + dy2*c], -1)



def silu_grad(u):

    sig = 1 / (1 + np.exp(-u))

    return sig * (1 + u * (1 - sig))



def loss_and_grads(gpt, ids, targets, mask=None):

    """23 课整机反向 + 25 课可选 mask(监督位加权)。mask=None 等价 23 课原版。"""

    T = len(ids)

    if mask is None:

        mask = [1.0] * T

    marr = np.array(mask, dtype=float)

    Ns = marr.sum()

    pos = np.arange(T)

    tape = []

    x = gpt.E[list(ids)]

    for W in gpt.layers:

        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']

        m = np.triu(np.ones((T, T), bool), 1)

        ps, outs = [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)

            s = np.where(m, -np.inf, s)

            p = softmax(s)

            ps.append(p); outs.append(p @ v[:, sl])

        co = np.concatenate(outs, -1)

        xm = x + co @ W['Wo']

        xr = gpt.rms(xm)

        u, zv = xr @ W['W1'], xr @ W['W3']

        gate = u/(1+np.exp(-u))*zv

        tape.append(dict(x_in=x, xm=xm, xr=xr, u=u, zv=zv, gate=gate, co=co, q=q, k=k, v=v, ps=ps))

        x = xm + gate @ W['W2']                 # 注意:x 入 tape 在覆盖之前(25 课战争故事②)



    xf = x

    logits = gpt.rms(xf) @ gpt.head

    z = logits - logits.max(-1, keepdims=True)

    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))

    loss = -(logp[np.arange(T), np.array(targets)] * marr).sum() / Ns



    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),

             'layers': [dict.fromkeys(W) for W in gpt.layers]}

    p = softmax(logits)

    onehot = np.zeros_like(p)

    onehot[np.arange(T), np.array(targets)] = 1.0

    dlogits = (p - onehot) * (marr / Ns)[:, None]      # ← mask 版与 23 课的核心差别

    grads['head'] = gpt.rms(xf).T @ dlogits

    dx = rms_backward(dlogits @ gpt.head.T, xf)

    for i in reversed(range(gpt.NL)):

        W, t = gpt.layers[i], tape[i]

        dg = dx @ W['W2'].T

        du = dg * silu_grad(t['u']) * t['zv']

        dzv = dg * (t['u']/(1+np.exp(-t['u'])))

        grads['layers'][i]['W2'] = t['gate'].T @ dx

        grads['layers'][i]['W1'] = t['xr'].T @ du

        grads['layers'][i]['W3'] = t['xr'].T @ dzv

        dxm = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['xm'])

        dc = dxm @ W['Wo'].T

        grads['layers'][i]['Wo'] = t['co'].T @ dxm

        dvs, dqs, dks = [], [], []

        for h in range(gpt.H):

            sl = slice(h*gpt.dh, (h+1)*gpt.dh)

            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]

            dO = dc[:, sl]

            dvs.append(p_h.T @ dO)

            dp = dO @ v_h.T

            ds = p_h * (dp - (dp*p_h).sum(-1, keepdims=True))

            dqs.append(ds @ k_h / math.sqrt(gpt.dh))

            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))

        dv = np.concatenate(dvs, -1)

        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)

        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)

        grads['layers'][i]['Wv'] = t['x_in'].T @ dv

        grads['layers'][i]['Wq'] = t['x_in'].T @ dq

        grads['layers'][i]['Wk'] = t['x_in'].T @ dk

        dx = dxm + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T

    np.add.at(grads['E'], np.array(ids), dx)

    return loss, grads



def flat_params(gpt):

    ps = [gpt.E, gpt.head]

    for W in gpt.layers:

        ps += [W[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return ps



def flat_grads(grads):

    gs = [grads['E'], grads['head']]

    for L in grads['layers']:

        gs += [L[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]

    return gs



def clip_norms(gs, max_norm=1.0):

    total = math.sqrt(sum((g ** 2).sum() for g in gs))

    if total > max_norm:

        s = max_norm / total

        for g in gs:

            g *= s

    return total



class MultiAdamW:

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):

        self.ps, self.lr, self.b1, self.b2, self.eps, self.wd = params, lr, *betas, eps, wd

        self.m = [np.zeros_like(p) for p in params]

        self.v = [np.zeros_like(p) for p in params]

        self.t = 0

    def step(self, gs):

        self.t += 1

        for p, g, m, v in zip(self.ps, gs, self.m, self.v):

            m *= self.b1; m += (1 - self.b1) * g

            v *= self.b2; v += (1 - self.b2) * g * g

            p -= self.lr * (m/(1-self.b1**self.t)/(np.sqrt(v/(1-self.b2**self.t))+self.eps) + self.wd*p)

def lr_at(step, total, base_lr=1e-3, warmup=100, floor_ratio=0.1):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base_lr * (floor_ratio + (1 - floor_ratio) * 0.5 * (1 + math.cos(math.pi * p)))

In [ ]:
# ===== 数据 + 模板 + 任务(讲义 §3、§5) =====

text = open("data/xiyouji.txt", encoding="utf-8").read()

chars = sorted(set(text))

stoi = {c: i for i, c in enumerate(chars)}

itos = {i: c for c, i in stoi.items()}

V0 = len(chars)

data = np.array([stoi[c] for c in text], dtype=np.int32)

n_train = int(len(data) * 0.99)

train_data, val_data = data[:n_train], data[n_train:]



T_USER, T_ASSI, T_END = V0, V0 + 1, V0 + 2



def enc(s):

    return [stoi[c] for c in s]



def extend_vocab(m, seed=1):

    r = np.random.default_rng(seed)

    m.E = np.vstack([m.E, r.normal(size=(3, m.D)) * 0.1])

    m.head = np.hstack([m.head, r.normal(size=(m.D, 3)) * 0.1])



ANTI = [("大","小"),("上","下"),("天","地"),("古","今"),("来","去"),("内","外"),

        ("高","低"),("长","短"),("黑","白"),("好","坏"),("真","假"),("生","死"),

        ("冷","热"),("快","慢"),("强","弱"),("轻","重"),("多","少"),("有","无"),

        ("明","暗"),("前","后"),("左","右"),("日","夜"),("开","关"),("爱","恨"),

        ("笑","哭"),("进","退"),("买","卖"),("胜","败"),("深","浅"),("忙","闲")]



COPY_STRINGS = ["金箍", "行者", "云端", "宝殿", "妖魔", "蒲团", "袈裟", "铁棒",

                "猴王", "灵山", "仙丹", "蟠桃", "佛祖", "天河", "石猴", "宝象",

                "法术", "神通", "筋斗", "火眼"]



def make_anti_ex(pair):

    q, a = pair

    ids = [T_USER] + enc(f"反义词:{q}") + [T_ASSI] + enc(a) + [T_END]

    mask = [0] * len(ids); mask[-2] = 1; mask[-1] = 1

    return ids, mask



def make_copy_ex(s):

    ids = [T_USER] + enc(f"重复:{s}") + [T_ASSI] + enc(s) + [T_END]

    mask = [0] * len(ids)

    for j in range(-len(s)-1, 0):

        mask[j] = 1

    return ids, mask



ALL_TASKS = [make_anti_ex(p) for p in ANTI] + [make_copy_ex(s) for s in COPY_STRINGS]

SPECIAL = {T_USER: '<|user|>', T_ASSI: '<|assistant|>', T_END: '<|end|>'}

def render(ids):

    return ''.join(SPECIAL[i] if i in SPECIAL else itos[i] for i in ids)

print(f"任务样本:{len(ANTI)} 反义词 + {len(COPY_STRINGS)} 复制 = {len(ALL_TASKS)}")

print("示例:", render(ALL_TASKS[0][0]), "| mask:", ALL_TASKS[0][1])

def val_loss_no_special(m):
    """全词表 val loss 与『去掉 3 个特殊 token 后在原词表上重归一化』的 val loss。"""
    rv = np.random.default_rng(7)
    off = rv.integers(0, len(val_data) - 65, 16)
    tot_full = tot_no = 0.0
    for o in off:
        xb = list(val_data[o:o+65])
        logits, _ = m.forward(xb[:-1])
        tgt = np.array(xb[1:])
        z = logits - logits.max(-1, keepdims=True)
        logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
        tot_full += -logp[np.arange(len(tgt)), tgt].mean()
        z2 = logits[:, :V0] - logits[:, :V0].max(-1, keepdims=True)
        logp2 = z2 - np.log(np.exp(z2).sum(-1, keepdims=True))
        tot_no += -logp2[np.arange(len(tgt)), tgt].mean()
    return tot_full / 16, tot_no / 16

In [ ]:
# ===== 权重泛化:mask → 逐 token 权重(讲义 §3) =====
def weighted_loss_and_grads(gpt, ids, targets, weights):
    """dlogits = (p−onehot)·w_t,即 loss=−Σ w·logp 的梯度。CE 是 w=mask/N 的特例。"""
    T = len(ids)
    w = np.array(weights, dtype=float)
    pos = np.arange(T)
    tape = []
    x = gpt.E[list(ids)]
    for W in gpt.layers:
        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']
        m = np.triu(np.ones((T, T), bool), 1)
        ps, outs = [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)
            s = np.where(m, -np.inf, s)
            p = softmax(s)
            ps.append(p); outs.append(p @ v[:, sl])
        co = np.concatenate(outs, -1)
        xm = x + co @ W['Wo']
        xr = gpt.rms(xm)
        u, zv = xr @ W['W1'], xr @ W['W3']
        gate = u/(1+np.exp(-u))*zv
        tape.append(dict(x_in=x, xm=xm, xr=xr, u=u, zv=zv, gate=gate, co=co, q=q, k=k, v=v, ps=ps))
        x = xm + gate @ W['W2']
    xf = x
    logits = gpt.rms(xf) @ gpt.head
    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),
             'layers': [dict.fromkeys(W) for W in gpt.layers]}
    p = softmax(logits)
    onehot = np.zeros_like(p)
    onehot[np.arange(T), np.array(targets)] = 1.0
    dlogits = (p - onehot) * w[:, None]
    grads['head'] = gpt.rms(xf).T @ dlogits
    dx = rms_backward(dlogits @ gpt.head.T, xf)
    for i in reversed(range(gpt.NL)):
        W, t = gpt.layers[i], tape[i]
        dg = dx @ W['W2'].T
        du = dg * silu_grad(t['u']) * t['zv']
        dzv = dg * (t['u']/(1+np.exp(-t['u'])))
        grads['layers'][i]['W2'] = t['gate'].T @ dx
        grads['layers'][i]['W1'] = t['xr'].T @ du
        grads['layers'][i]['W3'] = t['xr'].T @ dzv
        dxm = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['xm'])
        dc = dxm @ W['Wo'].T
        grads['layers'][i]['Wo'] = t['co'].T @ dxm
        dvs, dqs, dks = [], [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]
            dO = dc[:, sl]
            dvs.append(p_h.T @ dO)
            dp = dO @ v_h.T
            ds = p_h * (dp - (dp*p_h).sum(-1, keepdims=True))
            dqs.append(ds @ k_h / math.sqrt(gpt.dh))
            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))
        dv = np.concatenate(dvs, -1)
        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)
        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)
        grads['layers'][i]['Wv'] = t['x_in'].T @ dv
        grads['layers'][i]['Wq'] = t['x_in'].T @ dq
        grads['layers'][i]['Wk'] = t['x_in'].T @ dk
        dx = dxm + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T
    np.add.at(grads['E'], np.array(ids), dx)
    return grads

def seq_logprob(m, ids, n_ans):
    """答案段(最后 n_ans+1 个 token:答案各字 + <|end|>)的 log 概率之和。"""
    logits, _ = m.forward(ids[:-1])
    tg = np.array(ids[1:])
    z = logits - logits.max(-1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
    lp = logp[np.arange(len(tg)), tg]
    return lp[-(n_ans+1):].sum()

def add_grads(a, b):
    return {'E': a['E'] + b['E'], 'head': a['head'] + b['head'],
            'layers': [{k: L1[k] + L2[k] for k in L1}
                       for L1, L2 in zip(a['layers'], b['layers'])]}

def greedy_answer(m, prompt_ids, max_new=5):
    ids = list(prompt_ids)
    for _ in range(max_new):
        logits, _ = m.forward(ids)
        nxt = int(np.argmax(logits[-1]))
        if nxt == T_END:
            break
        ids = ids + [nxt]
    return ids

def task_acc(m):
    ok = 0; total = 0
    for q, a in [(f"反义词:{x}", y) for x, y in ANTI] + [(f"重复:{s}", s) for s in COPY_STRINGS]:
        prompt = [T_USER] + enc(q) + [T_ASSI]
        got = greedy_answer(m, prompt, len(a) + 1)[len(prompt):]
        ok += ("".join(itos.get(i, '#') for i in got) == a); total += 1
    return ok, total

In [ ]:
# ===== DPO 两遍法 + 数值验收(讲义 §3–4) =====
import copy as _copy

def dpo_step(pol, ref, ex_w, ex_l, beta=0.2):
    """一个偏好对上的 DPO。加权反向语义是 −Σw·logp,故
    chosen 权重 = +βσ(−βΔ),rejected 权重 = −βσ(−βΔ)。"""
    lp_w = seq_logprob(pol, *ex_w);  lp_l = seq_logprob(pol, *ex_l)
    lp_rw = seq_logprob(ref, *ex_w); lp_rl = seq_logprob(ref, *ex_l)
    delta = (lp_w - lp_rw) - (lp_l - lp_rl)
    loss = -math.log(1 / (1 + math.exp(-beta * delta)))
    sig_neg = 1 / (1 + math.exp(beta * delta))
    def ans_weights(ids, n, c):
        w = [0.0] * (len(ids) - 1)
        for j in range(1, n + 2):
            w[-j] = c
        return w
    g = add_grads(weighted_loss_and_grads(pol, ex_w[0][:-1], ex_w[0][1:],
                                          ans_weights(ex_w[0], ex_w[1],  beta * sig_neg)),
                  weighted_loss_and_grads(pol, ex_l[0][:-1], ex_l[0][1:],
                                          ans_weights(ex_l[0], ex_l[1], -beta * sig_neg)))
    return loss, delta, g

rng0 = np.random.default_rng(0)
m0 = MiniGPT(vocab=8, d=16, n_heads=2, n_layers=1, seed=3)
m0.E = np.vstack([m0.E, rng0.normal(size=(3, 16)) * 0.1])
m0.head = np.hstack([m0.head, rng0.normal(size=(16, 3)) * 0.1])
ref0 = _copy.deepcopy(m0)
ids_w = [3, 8, 5, 9, 4, 10]
ids_l = [3, 8, 5, 9, 6, 10]
_, _, g0 = dpo_step(m0, ref0, (ids_w, 1), (ids_l, 1))
def f():
    return dpo_step(m0, ref0, (ids_w, 1), (ids_l, 1))[0]
gmap = [g0['E'], g0['head']] + [W for L in g0['layers'] for W in L.values()]
amap = [m0.E, m0.head] + [W for L in m0.layers for W in L.values()]
errs = []
rr = np.random.default_rng(1)
for arr, gar in zip(amap, gmap):
    for _ in range(8):
        idx = tuple(rr.integers(0, s, 1)[0] for s in arr.shape)
        old = arr[idx]
        arr[idx] = old + 1e-6; fp = f()
        arr[idx] = old - 1e-6; fm = f()
        arr[idx] = old
        errs.append(abs(gar[idx] - (fp - fm) / 2e-6))
print(f"DPO 梯度抽查 {len(errs)} 坐标 max|err| = {max(errs):.2e}")

In [ ]:
# ===== 基座 + 故意欠训的弱 SFT(讲义 §5) =====
t0 = time.time()
gpt = MiniGPT(vocab=V0, d=64, n_heads=4, n_layers=2, seed=0)
params = flat_params(gpt)
opt = MultiAdamW(params)
rng = np.random.default_rng(42)
T, B = 64, 16
for step in range(2000):
    opt.lr = lr_at(step, 2000)
    off = rng.integers(0, len(train_data) - T - 1, B)
    gsum = None
    for b in range(B):
        xb = train_data[off[b]:off[b] + T + 1]
        loss, grads = loss_and_grads(gpt, list(xb[:-1]), list(xb[1:]))
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g / B for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g / B
    clip_norms(gsum, 1.0)
    opt.step(gsum)
extend_vocab(gpt)
print(f"[基座] 完成({time.time()-t0:.0f}s)")

sft = _copy.deepcopy(gpt)
ps = flat_params(sft)
osft = MultiAdamW(ps, lr=1e-4)
rr = np.random.default_rng(0)
for step in range(150):
    gsum = None
    idx = rr.integers(0, len(ALL_TASKS), 16)
    for j in idx:
        ids_, mk_ = ALL_TASKS[j]
        loss, grads = loss_and_grads(sft, ids_[:-1], ids_[1:], mk_[1:])
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g / 16 for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g / 16
    clip_norms(gsum, 1.0)
    osft.step(gsum)
acc_sft = task_acc(sft)
f_sft, _ = val_loss_no_special(sft)
print(f"[弱SFT 150步] 任务 {acc_sft[0]}/50 | 全词表 val {f_sft:.4f}")

In [ ]:
# ===== 偏好对 + DPO 训练器(讲义 §5) =====
QA = [(f"反义词:{x}", y) for x, y in ANTI] + [(f"重复:{s}", s) for s in COPY_STRINGS]
ALL_ANS = [a for _, a in QA]
rngp = np.random.default_rng(5)
pairs = []
for q, a in QA:
    prompt = [T_USER] + enc(q) + [T_ASSI]
    got = greedy_answer(sft, prompt, len(a) + 1)[len(prompt):]
    got_s = "".join(itos.get(i, '#') for i in got)
    if got_s != a and got_s and all(c in stoi for c in got_s):
        rej = got_s                      # 首选:模型自己的错误输出
    else:
        rej = ALL_ANS[rngp.integers(0, len(ALL_ANS))]
        while rej == a:
            rej = ALL_ANS[rngp.integers(0, len(ALL_ANS))]
    pairs.append(([T_USER] + enc(q) + [T_ASSI] + enc(a) + [T_END], len(a),
                  [T_USER] + enc(q) + [T_ASSI] + enc(rej) + [T_END], len(rej)))

def margins(pol, ref_m):
    ds = []
    for w, nw, l, nl in pairs:
        ds.append((seq_logprob(pol, w, nw) - seq_logprob(ref_m, w, nw))
                  - (seq_logprob(pol, l, nl) - seq_logprob(ref_m, l, nl)))
    return np.array(ds)

ref = _copy.deepcopy(sft)

def train_dpo(pol, ref_m, steps=200, lr=5e-5, beta=0.5, seed=0, log_every=0):
    rrr = np.random.default_rng(seed)
    ps2 = flat_params(pol)
    o2 = MultiAdamW(ps2, lr=lr)
    ref_lp = [(seq_logprob(ref_m, w, nw), seq_logprob(ref_m, l, nl)) for w, nw, l, nl in pairs]
    w_hist = []
    for step in range(steps):
        gsum = None
        tot = 0.0
        idx = rrr.integers(0, len(pairs), 8)
        for k in idx:
            w, nw, l, nl = pairs[k]
            delta = ((seq_logprob(pol, w, nw) - ref_lp[k][0])
                     - (seq_logprob(pol, l, nl) - ref_lp[k][1]))
            sig_neg = 1 / (1 + math.exp(beta * delta))
            tot += -math.log(1 / (1 + math.exp(-beta * delta)))
            w_hist.append(sig_neg)
            def aw(ids, n, c):
                ww = [0.0] * (len(ids) - 1)
                for j in range(1, n + 2):
                    ww[-j] = c
                return ww
            g = add_grads(weighted_loss_and_grads(pol, w[:-1], w[1:], aw(w, nw, beta * sig_neg)),
                          weighted_loss_and_grads(pol, l[:-1], l[1:], aw(l, nl, -beta * sig_neg)))
            gs = flat_grads(g)
            if gsum is None:
                gsum = [x / 8 for x in gs]
            else:
                for a2, x in zip(gsum, gs):
                    a2 += x / 8
        clip_norms(gsum, 1.0)
        o2.step(gsum)
        if log_every and (step + 1) % log_every == 0:
            print(f"    step {step+1}: DPO loss {tot/8:.4f}")
    return np.array(w_hist)

In [ ]:
# ===== 主实验:温和 / 激进 / β 遍历 / 弱基座对照(讲义 §5) =====
print("[DPO 温和] β=0.5 lr=5e-5 200 步:")
pol = _copy.deepcopy(sft)
w_hist = train_dpo(pol, ref, steps=200, lr=5e-5, beta=0.5, log_every=100)
acc_dpo = task_acc(pol)
f_dpo, _ = val_loss_no_special(pol)
d1 = margins(pol, ref)
print(f"  任务 {acc_sft[0]}/50 → {acc_dpo[0]}/50 | val {f_sft:.4f} → {f_dpo:.4f}"
      f" | Δ 均值 {d1.mean():+.3f} | Δ>0 占比 {(d1>0).mean():.0%}")
print(f"  难例权重 σ(−βΔ):前半 {w_hist[:len(w_hist)//2].mean():.3f}"
      f" → 后半 {w_hist[len(w_hist)//2:].mean():.3f}")

print("\n[DPO 激进] β=0.2 lr=1e-4 500 步(过优化警示):")
pol_hard = _copy.deepcopy(sft)
train_dpo(pol_hard, ref, steps=500, lr=1e-4, beta=0.2, log_every=250)
print(f"  任务 {acc_sft[0]}/50 → {task_acc(pol_hard)[0]}/50"
      f" | val {val_loss_no_special(pol_hard)[0]:.4f}"
      f" | Δ 均值 {margins(pol_hard, ref).mean():+.3f}")

print("\n[β 遍历] 200 步 lr=5e-5:")
for beta_try in (0.1, 0.5, 1.0):
    p2 = _copy.deepcopy(sft)
    train_dpo(p2, ref, steps=200, lr=5e-5, beta=beta_try)
    print(f"  β={beta_try}: 任务 {task_acc(p2)[0]}/50"
          f" | Δ 均值 {margins(p2, ref).mean():+.3f} | val {val_loss_no_special(p2)[0]:.4f}")

print("\n[对照] 400 步弱基座 + 弱 SFT + 温和 DPO(β=1.0):")
gq = MiniGPT(vocab=V0, d=64, n_heads=4, n_layers=2, seed=0)
pq = flat_params(gq)
oq = MultiAdamW(pq)
rq = np.random.default_rng(42)
for step in range(400):
    oq.lr = lr_at(step, 2000)
    off = rq.integers(0, len(train_data) - T - 1, B)
    gsum = None
    for b in range(B):
        xb = train_data[off[b]:off[b] + T + 1]
        loss, grads = loss_and_grads(gq, list(xb[:-1]), list(xb[1:]))
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g / B for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g / B
    clip_norms(gsum, 1.0)
    oq.step(gsum)
extend_vocab(gq)
sft_q = _copy.deepcopy(gq)
psq = flat_params(sft_q)
osq = MultiAdamW(psq, lr=1e-4)
rrq = np.random.default_rng(0)
for step in range(150):
    gsum = None
    idx = rrq.integers(0, len(ALL_TASKS), 16)
    for j in idx:
        ids_, mk_ = ALL_TASKS[j]
        loss, grads = loss_and_grads(sft_q, ids_[:-1], ids_[1:], mk_[1:])
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g / 16 for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g / 16
    clip_norms(gsum, 1.0)
    osq.step(gsum)
acc_q0 = task_acc(sft_q)
# 用 sft_q 的错误重建偏好对
pairs = []
for q_, a_ in QA:
    prompt = [T_USER] + enc(q_) + [T_ASSI]
    got = greedy_answer(sft_q, prompt, len(a_) + 1)[len(prompt):]
    got_s = "".join(itos.get(i, '#') for i in got)
    if got_s != a_ and got_s and all(c in stoi for c in got_s):
        rej = got_s
    else:
        rej = ALL_ANS[rngp.integers(0, len(ALL_ANS))]
        while rej == a_:
            rej = ALL_ANS[rngp.integers(0, len(ALL_ANS))]
    pairs.append(([T_USER] + enc(q_) + [T_ASSI] + enc(a_) + [T_END], len(a_),
                  [T_USER] + enc(q_) + [T_ASSI] + enc(rej) + [T_END], len(rej)))
ref_q = _copy.deepcopy(sft_q)
pol_q = _copy.deepcopy(sft_q)
train_dpo(pol_q, ref_q, steps=200, lr=5e-5, beta=1.0)
print(f"  弱起点:弱SFT {acc_q0[0]}/50 → DPO后 {task_acc(pol_q)[0]}/50")

print("\n[抽样]")
for q in ["反义词:大", "重复:筋斗"]:
    prompt = [T_USER] + enc(q) + [T_ASSI]
    for name, mm in [("弱SFT", sft), ("DPO后", pol)]:
        got = greedy_answer(mm, prompt, 5)[len(prompt):]
        print(f"  [{name}] {q} → " + repr("".join(itos.get(i, '<|end|>') for i in got)))

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 默写 dpo_step 两遍法,过数值裁判

# TODO(C2): seq_logprob 与 3 token 手算对照

# TODO(D1): 7B DPO 显存账:两份 bf16 权重;参考模型为什么不需要梯度

# TODO(D3): 查证 KTO / iterative DPO

# TODO(E1): rejected 全换随机错答重跑主表,与「模型自己的错误」对照

# TODO(E2/预习): 规则可判分的任务里,GRPO 的「组内基线」怎么替掉价值网络?
